# Last Mile Auditor — Veridi Logistics
**Question:** Are we failing specific regions, or is this nationwide?  
**Data:** Olist Brazilian e-commerce (orders, reviews, customers, products). Raw CSVs are downloaded at runtime into `data/raw/` — do not commit them.

Covers User Stories 1–4, the English-category bonus, and Candidate's Choice (promised vs actual transit by state).


In [ ]:
import sys
from pathlib import Path

# Colab only opens the .ipynb, not the repo. Clone it first so analyze.py and data/ exist.
if 'google.colab' in sys.modules:
    !pip -q install pandas plotly
    root = Path('/content/AmaliTech-DEG-Project-based-challenges/data-engineering/Logistics-auditor')
    if not (root / 'scripts' / 'analyze.py').exists():
        !git clone --depth 1 https://github.com/Jovz19200/AmaliTech-DEG-Project-based-challenges.git
    %cd /content/AmaliTech-DEG-Project-based-challenges/data-engineering/Logistics-auditor


In [1]:
from pathlib import Path
import json, sys
sys.path.insert(0, str(Path.cwd() / 'scripts' if (Path.cwd() / 'scripts').exists() else Path.cwd().parent / 'scripts'))
from analyze import build
import pandas as pd
import plotly.express as px

audit, summary = build()
pd.Series(summary['drop_log'])

orders_in                           99441
reviews_in                          99224
review_rows_dropped_as_duplicate      551
canceled_or_unavailable              1234
other_not_delivered                  1737
delivered_audited                   96470
master_rows                         99441
orders_duplicated_after_join            0
dtype: int64

## Story 1 — Schema builder

Join path: **Orders 1:1 latest Review**, **Orders many:1 Customers**, **Orders 1:1 first order item → product → English category**.

Reviews are 1-to-many. We keep the **latest review per `order_id`** so a second review cannot duplicate a shipment. After the joins, `orders_duplicated_after_join` in the drop log must be 0.


In [2]:
print('master grain check — duplicated orders after join:', summary['drop_log']['orders_duplicated_after_join'])
print('reviews dropped as extras:', summary['drop_log']['review_rows_dropped_as_duplicate'])
audit[['order_id','customer_state','review_score','Days_Difference','delivery_status','category_en']].head()

master grain check — duplicated orders after join: 0
reviews dropped as extras: 551


,order_id,customer_state,review_score,Days_Difference,delivery_status,category_en
0,e481f51cbdc54678b7cc49136f2d6af7,SP,4.0,7,On Time,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,BA,4.0,5,On Time,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,GO,5.0,17,On Time,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,RN,5.0,12,On Time,pet shop
4,ad21c59c0840e6cb83a9ceb5573f8159,SP,5.0,9,On Time,stationery


## Story 2 — The real delay calculator

`Days_Difference = estimated delivery − actual delivery` (in days).

- **On Time:** `Days_Difference >= 0` (arrived on or before the promise)
- **Late:** `−5 ≤ Days_Difference < 0`
- **Super Late:** `Days_Difference < −5`

Canceled and unavailable orders are excluded from delay rates (they never arrived, so they are not “late” — they are a different failure).


In [3]:
print(audit['delivery_status'].value_counts())
print('late %', summary['pct_late'], 'super late %', summary['pct_super_late'])
print('mean Days_Difference (positive = early)', summary['avg_days_difference'])
pd.DataFrame(summary['status_table'])

delivery_status
On Time       88644
Super Late     4211
Late           3615
Name: count, dtype: int64
late % 8.1 super late % 4.4
mean Days_Difference (positive = early) 10.88


,delivery_status,n,avg_review,avg_delay
0,On Time,88644,4.294080,-12.707177
1,Late,3615,3.460482,2.445920
2,Super Late,4211,1.784999,16.237473


## Story 3 — Geographic heatmap (bar)

São Paulo is the commercial hub. Remote = North + Northeast. If remote states are disproportionately late, the CEO’s “nationwide” gut feeling is wrong — it is a regional last-mile problem.


In [4]:
state = pd.DataFrame(summary['state_table'])
fig = px.bar(state.sort_values('pct_late', ascending=False), x='customer_state', y='pct_late', color='region',
             title='% of delivered orders that missed the ETA',
             labels={'pct_late':'% late','customer_state':'State'}, hover_data=['n','avg_review'])
fig.show()
print('worst', summary['worst_state'], summary['worst_state_pct_late'])
print('SP hub', summary['sp_pct_late'], 'remote N/NE', summary['remote_pct_late'])
pd.DataFrame(summary['region_table'])

worst AL 23.9
SP hub 5.9 remote N/NE 13.6


,region,n,pct_late,avg_review
0,Center-West,5624,8.0,4.127637
1,North,1796,9.8,4.031478
2,Northeast,9044,14.3,3.969805
3,South,13813,7.1,4.191415
4,Southeast,66193,7.5,4.179611


## Story 4 — Sentiment correlation

If late deliveries cause bad reviews, on-time scores should beat late scores, and the score should fall as delay grows.


In [5]:
print('on time', summary['on_time_review'], 'late', summary['late_review'], 'gap', summary['review_gap'])
sent = pd.DataFrame(summary['sentiment_table'])
fig = px.bar(sent, x='delay_bucket', y='avg_review', title='Average review vs delay bucket')
fig.show()
sample = audit.sample(min(len(audit), 8000), random_state=7)
fig2 = px.scatter(sample, x='delay_days', y='review_score', color='region', opacity=0.35,
                  title='Days late vs review score')
fig2.show()

on time 4.29 late 2.57 gap 1.73


## Bonus — English categories

`product_category_name` is Portuguese. We map it with `product_category_name_translation.csv` so furniture vs electronics is readable.


In [6]:
cat = pd.DataFrame(summary['category_table'])
fig = px.bar(cat.sort_values('pct_late', ascending=False).head(12), x='category_en', y='pct_late',
             title='Late rate by English category (n ≥ 200)')
fig.update_layout(xaxis_tickangle=-30)
fig.show()
cat.head(10)

,category_en,n,pct_late,avg_review
0,audio,344,13.1,3.847507
1,books technical,255,11.0,4.425197
2,home confort,370,10.3,3.907609
3,food,435,10.1,4.324826
4,electronics,2507,9.9,4.128617
5,baby,2763,9.3,4.121489
6,construction tools construction,728,9.2,4.132780
7,office furniture,1246,9.2,3.650485
8,construction tools lights,231,9.1,4.214912
9,health beauty,8608,9.0,4.235459


## Candidate's choice — promised vs actual transit

The CEO asked whether we **over-promise**. Plot average promised days (purchase → ETA) against average actual days (purchase → delivered) by state. Points **above the diagonal** take longer than we told the customer. That is an ETA policy fix, not a truck fix.


In [7]:
state = pd.DataFrame(summary['state_table'])
fig = px.scatter(state, x='avg_promised', y='avg_actual', size='n', color='region', hover_name='customer_state',
                 title='Promised vs actual transit days')
mx = max(state['avg_promised'].max(), state['avg_actual'].max())
fig.add_shape(type='line', x0=0, y0=0, x1=mx, y1=mx, line=dict(dash='dash', color='#888'))
fig.show()
print(summary['recommendation'])

Late deliveries are regional, not nationwide: AL is at 23.9% late vs São Paulo at 5.9%. On-time orders score 4.29/5; late orders score 2.57/5 (a 1.73 point drop). North/Northeast run 13.6% late. Repair ETAs and last-mile capacity there first.
